In [1]:
import os
import certifi
import requests
from dotenv import load_dotenv
from langchain.tools import tool

from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults


In [2]:
!pip install langchainhub



In [3]:
from langchain import hub
from langchain.agents import create_react_agent ,AgentExecutor


In [4]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GROQ_API_KEY   = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [5]:
search_tool = TavilySearchResults(max_results=5)

In [6]:


@tool
def get_weather(city: str) -> str:
    """
    Get the current weather for a given city using the Weatherstack API.
    """
    if not WEATHERSTACK_API_KEY:
        return "Weatherstack API key is not set."

    url = f"http://api.weatherstack.com/current?access_key={WEATHERSTACK_API_KEY}&query={city}"
    response = requests.get(url)
    
    if response.status_code != 200:
        return f"Error: Unable to fetch weather data. Status code: {response.status_code}"

    data = response.json()
    
    if "error" in data:
        return f"Error: {data['error']['info']}"

    current_weather = data.get("current", {})
    temperature = current_weather.get("temperature")
    weather_descriptions = current_weather.get("weather_descriptions", [])
    
    if temperature is None or not weather_descriptions:
        return "Weather information is not available."

    description = ", ".join(weather_descriptions)
    return f"The current temperature in {city} is {temperature}°C with {description}."

In [7]:
result = search_tool.invoke("What is AI")
print(result)


[{'url': 'https://en.wikipedia.org/wiki/Artificial_intelligence', 'content': 'Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.\n\nHigh-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go "Go (game)")), and content generation (e.g. images, audio, and videos).\n\nThe traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robo

In [8]:
llm = ChatGroq(api_key=GROQ_API_KEY, model="openai/gpt-oss-120b")

In [9]:
result = llm.invoke("What is Recursive language model?")
print(result)

content='**Recursive language model – a quick‑look overview**\n\n---\n\n### 1. What the term means\n\nA **recursive language model (RLM)** is a neural model that builds representations of a sentence (or any text span) by **recursively combining the representations of its sub‑parts according to a hierarchical structure**—usually a syntactic parse tree.  \n\nIn other words, instead of processing a sequence strictly left‑to‑right (as a standard recurrent neural network or transformer does), an RLM:\n\n1. **Starts at the leaves** (words or tokens) of a tree.  \n2. **Applies a composition function** (often a small feed‑forward network) to two child nodes to produce a parent node’s vector.  \n3. **Repeats** this composition up the tree until a single root vector represents the whole sentence.\n\nThat root vector can then be used for any downstream language‑modeling task (next‑word prediction, masked‑word filling, classification, etc.).  \n\n> **Key idea:** *Structure‑driven composition* → th

In [10]:
prompt = hub.pull("hwchase17/react")

/home/dharmasugash/Desktop/AgenticAI_Course/.venv/lib/python3.11/site-packages/langchain/hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [11]:
prompt


PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [12]:
tools = [search_tool,get_weather]


In [13]:
agent = create_react_agent(
    llm=llm, 
    tools=tools,
     prompt=prompt
     )

In [14]:
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=False
)

In [15]:
result = agent_executor.invoke({
    "input":(
    "tell me the weather in New York "
    "and also tell me what is AI"
    )
    })



/home/dharmasugash/Desktop/AgenticAI_Course/.venv/lib/python3.11/site-packages/langchain_groq/chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
/home/dharmasugash/Desktop/AgenticAI_Course/.venv/lib/python3.11/site-packages/langchain_groq/chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
/home/dharmasugash/Desktop/AgenticAI_Course/.venv/lib/python3.11/site-packages/langchain_groq/chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://

ValueError: An output parsing error occurred. In order to pass this error back to the agent and have it try again, pass `handle_parsing_errors=True` to the AgentExecutor. This is the error: Could not parse LLM output: `**Weather in New York (current):**  
- Temperature: **22 °C**  
- Condition: **Clear**

**What is AI?**  
Artificial Intelligence (AI) is a branch of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. These tasks include learning from data (machine learning), recognizing speech or images, understanding natural language, making decisions, and solving problems. AI systems range from simple rule‑based programs to advanced deep‑learning models that can generate text, drive cars, diagnose medical conditions, and more. In essence, AI aims to enable machines to perceive, reason, and act autonomously or with minimal human guidance.`

In [16]:
print(result["output"])

TypeError: 'AIMessage' object is not subscriptable